In [ ]:
import anthropic 
import mysql.connector
from dotenv import load_dotenv
import base64

import json
import os
load_dotenv()

False

In [ ]:
#FUNCTION: Connects to the LLM. Scans the respected pdf. Translates handwritten data into dictionary. Confirms confidence of scan.
client = anthropic.Anthropic()
def read_trf(pdf):
    #Convert PDF into base64
    with open(pdf, "rb") as file:
        pdf_bytes = file.read()
    encoded_bytes = base64.b64encode(pdf_bytes)
    pdf_base64_string = encoded_bytes.decode("utf-8")

    #Retry Loop: call Claude, check confidence, retry up to 3x
    extracted_data = None
    flagged_retry = 0
    while flagged_retry < 3:
        if extracted_data is None:
            return {"status": "failed", "reason": "Could not parse a valid response after 3 attempts"}
        client_message = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=8000,
         messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "document",
                        "source": {
                            "type": "base64",
                            "media_type": "application/pdf",
                            "data": pdf_base64_string
                        }
                    },
                    {
                        "type": "text",
                        "text": (
                            "Extract the handwritten print under each respected field. "
                            "Ordering Physician Information: Practice/Client Name, Phone, Last Name, First Name, NPI # "
                            "Email, Fax, Street Address, Suite/Bldg #, City, State, Postal Code "
                            "Patient Information: Last Name Required, First Name Required, MI, Date Of Birth(MM/DD/YYYY), Phone, Sex at Birth, "
                            "Email(for online report access), Medical Record #, Street Address/PO Box/Building/Floor/etc, City, State/Province/Region, Zip/Postal Code/ Country "
                            "Check which box is checked off, whichever has the majority mark is the one selected, there can only be one. "
                            "Demographic: Race: White, Asian, Black/African American, Native America/ Alaska Native, Native Hawaiian/ Other Pacific Islander, "
                            "Ethnicity: Not Hispanic, Latino, Spanish OR Hispanic, Latino, Spanish "
                            "Patient History: Diabetes AND/OR Family of MI Z82.49, Family History of Heart Attack, Stroke, Coronary Artery Bypass, Stent or Angina, <= "
                            "65 years of age(Parent/sibling/child) AND/OR High Dose Biotin "
                            "Billing Information: Client OR Patient Self-Pay, Test Requested: 1003 SmartVascular DX (SVDx) OR 1028 Expanded Lipid Profile(ELP) OR "
                            "1003/1028 Smart Vascular DX and ELP Panel. Specimen Collection: Date Collected in MM/DD/YYYY, Time (AM or PM marked), Collected by(please print) "
                            "Ordering Physician: Just note if signature is there then say: Signature collected, if not say Signature not collected, Date: in MM/DD/YYYY "
                            "Patient Acknowledgment: Just note if signature is there then say: Signature collected, if not say Signature not collected, Date: in MM/DD/YYYY. "
                            "Once carefully extracting all fields specified, return all extracted data as a structured JSON object using these extracted field names, and include a "
                            "confidence score between 0 and 1 for each field. Keep all original fields and values in the main structured JSON as specified above. "
                            "Use lowercase snake_case field names as the JSON keys (for example: first_name, date_of_birth, npi). " 
                            "Additionally, create a separate JSON object called 'low_confidence_fields' that includes the field name and value for any field where your confidence score is below 0.9."
                        )
                    }
                ]
            }
        ]
    )
        #Claude often wraps its JSON in markdown backticks ('''JSON''') 
        #Strip them before parsing so json.load() doesnt crash on the markdown backtick
        response_text = client_message.content[0].text
        response_text = response_text.strip()
        response_text = response_text.removeprefix("```json").removeprefix("```")
        response_text = response_text.removesuffix("```")
        response_text = response_text.strip()

        try: 
            extracted_data = json.loads(response_text)
        except json.JSONDecodeError:
            flagged_retry += 1
            continue

        #Only runs if parsing successful
        if not extracted_data["low_confidence_fields"]:
            break

            flagged_retry += 1
    if extracted_data["low_confidence_fields"]:
        extracted_data["permanently_flagged"] = extracted_data["low_confidence_fields"]

    return extracted_data



SyntaxError: unterminated string literal (detected at line 10) (1700352736.py, line 10)